In [1]:
import sys
sys.path.append('../')

import numpy as np
import scipy.sparse as ssp
import matplotlib.pyplot as plt
import qutip as qt
import scqubits as scq
from matplotlib.colors import LogNorm
from tqdm import tqdm
from qutip.qip.operations import rz, cz_gate
import cmath
from sympy import symbols
import utils_2Q_gate_zp as ut
ut.set_fig_font() ### Set various sizes in plotting
import scipy as sp
from joblib import Parallel, delayed
import itertools
from qutip.qip.operations import rz, cz_gate, cnot, rx, hadamard_transform, swap
import pandas as pd

### Get fidelity for input params

In [2]:
truc1, truc_tot, charge_pick = 300, 1000, True
truc_full = 100

folder = f'../../data/3ncut_two_zeropi/truc1={truc1}_truc2={truc_tot}_pick={charge_pick}/'
eval_tot = 2*np.pi* pd.read_csv(folder+ 'eval_tot.txt').to_numpy().flatten()
n_theta0_dress = 2*np.pi* np.load(folder+'n_theta0_dress.npy')
hspace_full = pd.read_csv(folder+ 'hspace_full.txt').to_numpy().flatten().tolist()

truc_list = np.arange(truc_full)
hspace_full = hspace_full[:truc_full]
eval_tot = eval_tot[:truc_full]
n_theta0_dress = ut.truncate_2(n_theta0_dress, truc_list)

logi_state = ['0-0', '0-2', '2-0', '2-2']
idx_0 = hspace_full.index('0-2')
idx_1 = hspace_full.index('2-2')
idx_2 = hspace_full.index('8-2')
W_0_2 = eval_tot[idx_2] - eval_tot[idx_0]
W_1_2 = eval_tot[idx_2] - eval_tot[idx_1]

num_cpus = 16
x0_vec = np.array([
[19.98707454, 0.27836391, 0.17709104, -0.00011593, -0.00057114]
])
n_job = 100
c_op_list = []

H0_full = qt.Qobj(np.diag(eval_tot))
logi_idx_full = [hspace_full.index(i) for i in logi_state]
H_drive_full = [ H0_full,   [n_theta0_dress, ut.drive_gauss_A],
                            [n_theta0_dress, ut.drive_gauss_B]  ]

In [3]:
hspace_select = [
'8-2', '0-0', '0-2', '2-0', '2-2',
]
index_select = [hspace_full.index(i) for i in hspace_select]
len_select = len(hspace_select)
H0_select = ut.truncate_2( H0_full, index_select)
n_theta0_select = ut.truncate_2(n_theta0_dress, index_select)
logi_idx_select = [hspace_select.index(i) for i in logi_state]
H_drive_select = [ H0_select,   [n_theta0_select, ut.drive_gauss_A],
                                [n_theta0_select, ut.drive_gauss_B]  ]

arg_select = [H_drive_select, W_0_2, W_1_2, num_cpus, c_op_list, logi_idx_select]
f_select = Parallel(n_jobs=n_job)(delayed(ut.cnot_fidelity_log_tg)(args_indep, *arg_select)
                                            for args_indep in x0_vec)
print(f' fidelity (dim={len(hspace_select)},{charge_pick}) =', np.round(f_select, 8).tolist())
# print(f'fidelity (dim={len(hspace_truc)},{charge_pick}) =', np.round(f_truc, 8).tolist())
# print(f'fidelity (dim={truc_full},{charge_pick}) =', np.round(f_full, 8).tolist())

succeed
Quantum object: dims = [[4], [4]], shape = (4, 4), type = oper, isherm = False
Qobj data =
[[ 1.00000000e+00+5.36609441e-10j  1.28526269e-07+8.06632152e-07j
  -2.02844755e-09+2.77744789e-09j -1.25586749e-07-8.05573975e-07j]
 [ 4.74863986e-06+1.31810540e-06j -6.58764997e-04+6.46330938e-04j
   1.77910246e-03-1.91091515e-04j -1.36527765e-01-9.90634281e-01j]
 [ 4.43009124e-10-1.01170905e-09j  1.74029151e-03+4.47035076e-04j
  -4.64798302e-01-8.85410530e-01j  1.55863652e-03-8.91876496e-04j]
 [ 3.75070130e-06-7.17751363e-06j -6.17463513e-01-7.86596743e-01j
   2.90214408e-04-1.76307925e-03j  7.76379191e-04-1.55511245e-05j]]
 fidelity (dim=5,True) = [-5.42028514]


In [ ]:
=

In [ ]:
hspace_truc = hspace_full[:190]
truc_index = [hspace_full.index(i) for i in hspace_truc]
truc_len = len(hspace_truc)
H0_truc = ut.truncate_2( H0_full, truc_index)
n_theta0_truc = ut.truncate_2(n_theta0_dress, truc_index)
logi_idx_truc = [hspace_truc.index(i) for i in logi_state]
H_drive_truc = [ H0_truc,   [n_theta0_truc, ut.drive_gauss_A],
                            [n_theta0_truc, ut.drive_gauss_B]  ]

arg_truc = [H_drive_truc, W_0_2, W_1_2, num_cpus, c_op_list, logi_idx_truc]
f_truc = Parallel(n_jobs=n_job)(delayed(ut.cnot_fidelity_log_tg)(args_indep, *arg_truc)
                                            for args_indep in x0_vec)
print(f' fidelity (dim={len(hspace_truc)},{charge_pick}) =', np.round(f_truc, 8).tolist())

 fidelity (dim=190,True) = [-0.22261318]


In [ ]:
arg_full = [H_drive_full, W_0_2, W_1_2, num_cpus, c_op_list, logi_idx_full]
f_full = Parallel(n_jobs=n_job)(delayed(ut.cnot_fidelity_log_tg)(args_indep, *arg_full)
                                            for args_indep in x0_vec)
# print(f' fidelity (dim={len(hspace_truc)},{charge_pick}) =', np.round(f_truc, 8).tolist())
print(f'fidelity (dim={truc_full},{charge_pick}) =', np.round(f_full, 8).tolist())

fidelity (dim=1000,True) = [-2.39932723]


In [ ]:
hspace_select = [
'0-0', '0-2', '2-0', '8-2', '2-2', '12-2', '4-9', '1-2', '1-0', '5-2' ,
'5-0', '8-5', '22-2', '9-2', '2-1', '5-8', '5-5', '0-1', '34-2', '2-5' ,
'13-2', '4-2', '8-0', '1-1', '0-5', '8-9', '15-2', '26-2', '30-2', '1-4' ,
'9-0', '20-2', '20-5', '12-5', '4-0', '15-5', '5-1', '18-2', '12-0', '24-2' ,
'26-5', '1-5', '15-0', '13-0', '25-2', '35-2', '8-1', '33-2', '9-5', '4-5' ,

'12-8', '9-8', '1-8', '37-2', '5-12', '25-0', '9-4', '1-25', '22-5', '56-2' ,
'18-4', '13-9', '4-4', '2-4', '15-4', '50-2', '13-5', '9-9', '5-9', '4-1' ,
'20-0', '25-5', '15-9', '18-5', '9-1', '30-5', '2-21', '26-9', '1-13', '20-9' ,
'45-2', '18-0', '5-16', '22-0', '34-5', '15-1', '5-4', '0-21', '20-4', '26-0' ,
'25-4', '25-1', '0-4', '9-12', '24-5', '37-5', '24-0', '1-18', '12-9', '44-2' ,

'25-8', '2-12', '41-2', '4-16', '4-12', '4-21', '8-12', '18-9', '18-1', '45-0' ,
'24-9', '26-8', '39-2', '4-8', '8-4', '34-0', '22-8', '35-4', '12-4', '30-0' ,
'12-1', '28-2', '18-8', '0-8', '35-5', '35-0', '9-16', '37-0', '2-8', '1-9' ,
'13-8', '46-2', '2-26', '46-0', '2-9', '41-0', '4-13', '39-0', '2-25', '0-12' ,
'22-9', '33-0', '33-4', '22-4', '54-2', '8-18', '28-1', '0-45', '50-0', '24-1' ,

'28-5', '30-1', '15-8', '34-1', '22-1', '20-1', '13-16', '12-18', '0-9', '24-8' ,
'13-1', '51-2', '33-5', '2-30', '1-21', '4-18', '59-0', '15-12', '0-25', '8-24' ,
'8-8', '9-18', '44-0', '26-1', '65-0', '56-0', '41-1', '41-4', '5-30', '39-4' ,
'30-4', '58-0', '5-13', '18-12', '0-30', '45-4', '0-39', '13-4', '46-1', '28-0' ,
'2-18', '0-24', '0-13', '9-13', '30-8', '69-0', '41-5', '33-1', '37-8', '4-24' ,
]
index_select = [hspace_full.index(i) for i in hspace_select]
len_select = len(hspace_select)
H0_select = ut.truncate_2( H0_full, index_select)
n_theta0_select = ut.truncate_2(n_theta0_dress, index_select)
logi_idx_select = [hspace_select.index(i) for i in logi_state]
H_drive_select = [ H0_select,   [n_theta0_select, ut.drive_gauss_A],
                                [n_theta0_select, ut.drive_gauss_B]  ]

arg_select = [H_drive_select, W_0_2, W_1_2, num_cpus, c_op_list, logi_idx_select]
f_select = Parallel(n_jobs=n_job)(delayed(ut.cnot_fidelity_log_tg)(args_indep, *arg_select)
                                            for args_indep in x0_vec)
print(f' fidelity (dim={len(hspace_select)},{charge_pick}) =', np.round(f_select, 8).tolist())
# print(f'fidelity (dim={len(hspace_truc)},{charge_pick}) =', np.round(f_truc, 8).tolist())
# print(f'fidelity (dim={truc_full},{charge_pick}) =', np.round(f_full, 8).tolist())

 fidelity (dim=200,True) = [-0.35647568, -0.81452097]
